In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
INGESTION_DIR = ROOT / 'services' / 'ingestion-service'
RAG_DIR = ROOT / 'services' / 'rag-service'

# Add service paths so we can import existing modules
sys.path.insert(0, str(INGESTION_DIR))
sys.path.insert(0, str(RAG_DIR))

print(f'Root: {ROOT}')
print(f'Ingestion path ok: {INGESTION_DIR.exists()}')
print(f'RAG path ok: {RAG_DIR.exists()}')

Root: c:\Users\KritChaJ\OneDrive\Documents\CPE CHAT 0.0.3
Ingestion path ok: True
RAG path ok: True


## 1) Embedding sanity check (reuse `scripts/test_bge_m3.py`)
- รันสคริปต์ที่มีอยู่เพื่อตรวจสอบ dimension=1024, instruction-based query embedding, typo/cross-lingual robustness
- ใช้ Thai spell correction ตามที่ pipeline มีอยู่

In [2]:
import runpy

print('Running existing script: services/ingestion-service/scripts/test_bge_m3.py')
_ = runpy.run_path(str(INGESTION_DIR / 'scripts' / 'test_bge_m3.py'))

Running existing script: services/ingestion-service/scripts/test_bge_m3.py


c:\Users\KritChaJ\OneDrive\Documents\CPE CHAT 0.0.3\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded BGE-M3 model: BAAI/bge-m3


## 2) Chunk size distribution (จาก `chunks.toon`)
- ใช้ heuristic 4 ตัวอักษร ≈ 1 token (เหมือนใน `rag_logic.est_tokens`)
- ดู p50/p90/p95 เพื่อปรับ `CHUNK_MIN_TOKENS`, `CHUNK_MAX_TOKENS`, `CHUNK_OVERLAP_RATIO`

In [3]:
from app.toon_converter import read_toon
import numpy as np

possible = [
    ROOT / 'data' / 'db' / 'chunks.toon',
    INGESTION_DIR / 'data' / 'db' / 'chunks.toon',
    INGESTION_DIR / 'data' / 'db' / 'data_chunks.toon',
]
chunks = []
chosen = None
for p in possible:
    if p.exists():
        data = read_toon(str(p))
        if isinstance(data, dict):
            chunk_list = data.get('chunks') or data.get('data') or []
        elif isinstance(data, list):
            chunk_list = data
        else:
            chunk_list = []
        chunks = chunk_list
        chosen = p
        break

if not chunks:
    print('No chunks file found; run ingestion first.')
else:
    print(f'Using chunks from: {chosen}')
    lengths = []
    for c in chunks:
        text = c.get('text', '')
        tokens = max(1, int(len(text) / 4))
        lengths.append(tokens)
    arr = np.array(lengths, dtype=float)
    stats = {
        'count': int(arr.size),
        'mean_tokens': float(arr.mean()),
        'median_tokens': float(np.median(arr)),
        'p90': float(np.percentile(arr, 90)),
        'p95': float(np.percentile(arr, 95)),
        'min': float(arr.min()),
        'max': float(arr.max()),
    }
    print('Token length stats:', stats)

No chunks file found; run ingestion first.


## 3) BM25 vs Vector vs Hybrid (RRF)
- ใช้ฟังก์ชันใน `rag_service.app`: `semantic_search`, `keyword_search`, `hybrid_retrieve`
- ดู top-k จากแต่ละวิธีและคะแนน `score_rrf` ที่ผสานแล้ว

In [4]:
from app.rag_logic import hybrid_retrieve
from app.chroma_client import semantic_search
from app.sqlite_client import keyword_search, fetch_docs

def compare_methods(question: str, k: int = 5):
    sem = semantic_search(question, top_k=k)
    kw_ids = keyword_search(question, limit=k)
    kw_docs = fetch_docs(kw_ids)
    hybrid = hybrid_retrieve(question, k_vec=k, k_kw=k)

    def summarize(items, label):
        print(f'
{label} (top {len(items)}):')
        for i, doc in enumerate(items, 1):
            src = doc.get('source') or doc.get('path') or doc.get('doc_id') or '?'
            page = doc.get('page_start', doc.get('page', '?'))
            score = doc.get('score') or doc.get('score_rrf') or doc.get('distance')
            print(f
)
    summarize(kw_docs, 'BM25 / keyword (SQLite FTS)')
    summarize(sem, 'Vector (Chroma)')
    summarize(hybrid, 'Hybrid (RRF merge)')
    return {'semantic': sem, 'keyword': kw_docs, 'hybrid': hybrid}

sample_questions = [
    'วิชาบังคับในหลักสูตรคืออะไร',
    'เปิดรับสมัครนักศึกษาเมื่อไหร่',
]

results = [compare_methods(q, k=5) for q in sample_questions]

SyntaxError: unterminated f-string literal (detected at line 12) (3909299560.py, line 12)

## 4) Human evaluation checklist
- สุ่มดู top-k contexts จาก `results` หรือ `rag_query()` แล้วให้มนุษย์ให้คะแนน (0-3) ด้านความเกี่ยวข้อง/ความครบถ้วน/ความถูกต้อง
- ใช้ผู้ประเมิน ≥2 คน แล้ววัด Cohen's kappa เพื่อตรวจสอบความสอดคล้อง
- ติดธงกรณีที่ hybrid ชนะ BM25/Vector เดี่ยวหรือกลับกัน เพื่อนำไปปรับ RRF_K, k_vec, k_kw
- ถ้าต้องการ template CSV ให้คัดลอกคอลัมน์: `question,retriever,top_k,relevance,completeness,accuracy,notes`